# Bronze Layer Process

In [ ]:
import pandas as pd 
import json
#Need to load in JSON dictionaries in order to work with pandas
#starting with one slice for testing 
with open("../data/mpd.slice.0-999.json") as f:
    data = json.load(f)

#This pandas method transforms the nested dictionaries into a flat table
pd.json_normalize(data)    

In [ ]:
#The following is for playlists
normaldata = pd.json_normalize(data, record_path="playlists") #record_path enables user to set path into 'playlists' to handle nested list
print(type(normaldata))
normaldata


In [ ]:
#The following took 17 minutes to finish - indicating that invoking to_csv on EVERY interation during the loop, the OS has to write (append) to SSD/HDD which is the biggest bottleneck

from pathlib import Path

datapath = Path('../data/') #Path containing all slices

header = True #Ensuring header is written only on the first iteration
with open ("../bronze/sliceinfo.csv", "a", newline="") as file:
    for i in datapath.iterdir():
        with open(i,'r') as f:
            jread = json.load(f)
        df = pd.json_normalize(jread)
        df.to_csv(file, header=header)
        header = False

In [ ]:
# Attempt will be to append to all data to a list (using RAM here), convert it to a dataframe and perform one write operation 
#Per pandas- do not use CONCAT repeatedly as every call makes a copy of the data

# Warning ! This crashed my Lenovo - not enough RAM
from pathlib import Path

spdatapath = Path("../data/")

df = pd.DataFrame() #instantiate empty dataframe 
df_list = []

for slice in spdatapath.iterdir(): #iterate through data directory (path object - posix in Lenovo)
    with open(slice,'r') as f: #opens slice on current iteration
        sliceread= json.load(f) #reads nested JSON
        df = pd.json_normalize(sliceread) #standardizes and convert JSON data of current slice to dataframe
        df_list.append(df) #appending 



In [ ]:
# May have reached bottle neck of working with csv files- computation is too expensive 

# Will use parquet file processing for analysis (using old csv logic) - PyArrow will be the engine used by pandas

import json
import pandas as pd
from pathlib import Path

datapath = Path('../data/') #Path containing all slices
bronze_playlist_path = Path ('../bronze/playlist')
bronze_playlist_path.mkdir(exist_ok=True)

bronze_sliceinfo_path = Path('../bronze/sliceinfo')
bronze_sliceinfo_path.mkdir(exist_ok=True)


for file in datapath.iterdir(): #iterate through directory using path .iterdir() method
    with open (file, 'r') as f: #open file on current iteration (going from path object to json file)
        jread=json.load(f) #loads the json
    df = pd.json_normalize(jread['playlists']) #normalizing at playlists level

    #standard naming convention
    out_name = file.stem.replace('mpd.slice.', 'slices_') + '.parquet' #follows name convention
    df.to_parquet(bronze_playlist_path / out_name) #parquet conversion and placed in bronze

#the above took 6 minuntes and 6 seconds

In [ ]:
for file in datapath.iterdir():
    with open (file,'r') as f:
        jread=json.load(f)
    df = pd.json_normalize([jread['info']]) #trying to normalize to the info level

    out_name = file.stem.replace('mpd.slice.', 'slices_') + '.parquet'
    df.to_parquet(bronze_sliceinfo_path / out_name)
    #took 3 mins 25 seconds

In [ ]:
#The following checkes for environment variables. This was used to troubleshoot pyspark 

import os
import sys

print("Python Executable:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print(spark.version)



In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SpotifyPipeLine") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .getOrCreate() # allowing all cores to be utilized with .master, .config to 16gb which is half of what my local setup has


In [38]:
dfp = spark.read.parquet("../bronze/playlist") #reads in the entire bronze directory ->Notice this takes 3 seconds

dfs = spark.read.parquet("../bronze/sliceinfo")

In [3]:
dfs.printSchema() #slice info raw schema

root
 |-- generated_on: string (nullable = true)
 |-- slice: string (nullable = true)
 |-- version: string (nullable = true)



In [4]:
dfp.printSchema()

root
 |-- name: string (nullable = true)
 |-- collaborative: string (nullable = true)
 |-- pid: long (nullable = true)
 |-- modified_at: long (nullable = true)
 |-- num_tracks: long (nullable = true)
 |-- num_albums: long (nullable = true)
 |-- num_followers: long (nullable = true)
 |-- tracks: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- album_name: string (nullable = true)
 |    |    |-- album_uri: string (nullable = true)
 |    |    |-- artist_name: string (nullable = true)
 |    |    |-- artist_uri: string (nullable = true)
 |    |    |-- duration_ms: long (nullable = true)
 |    |    |-- pos: long (nullable = true)
 |    |    |-- track_name: string (nullable = true)
 |    |    |-- track_uri: string (nullable = true)
 |-- num_edits: long (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- num_artists: long (nullable = true)
 |-- description: string (nullable = true)



# Silver Layer Process

In [5]:
dfp.count() #1000000 playlists checks out

1000000

In [6]:
dfs.count() #1000 slices also checks out

1000

In [7]:
dfs.show()

+--------------------+-------------+-------+
|        generated_on|        slice|version|
+--------------------+-------------+-------+
|2017-12-03 08:41:...|100000-100999|     v1|
|2017-12-03 08:41:...|101000-101999|     v1|
|2017-12-03 08:41:...|102000-102999|     v1|
|2017-12-03 08:41:...|103000-103999|     v1|
|2017-12-03 08:41:...|104000-104999|     v1|
|2017-12-03 08:41:...|105000-105999|     v1|
|2017-12-03 08:41:...|106000-106999|     v1|
|2017-12-03 08:41:...|107000-107999|     v1|
|2017-12-03 08:41:...|108000-108999|     v1|
|2017-12-03 08:41:...|109000-109999|     v1|
|2017-12-03 08:41:...|110000-110999|     v1|
|2017-12-03 08:41:...|111000-111999|     v1|
|2017-12-03 08:41:...|112000-112999|     v1|
|2017-12-03 08:41:...|113000-113999|     v1|
|2017-12-03 08:41:...|114000-114999|     v1|
|2017-12-03 08:41:...|115000-115999|     v1|
|2017-12-03 08:41:...|116000-116999|     v1|
|2017-12-03 08:41:...|117000-117999|     v1|
|2017-12-03 08:41:...|118000-118999|     v1|
|2017-12-0

In [8]:
dfp.show(10) #data is seemingly not in any particular order...info.slice shows various slices appearing in the first 10 rows... operation also took about 29 seconds

+-------------+-------------+------+-----------+----------+----------+-------------+--------------------+---------+-----------+-----------+-----------+
|         name|collaborative|   pid|modified_at|num_tracks|num_albums|num_followers|              tracks|num_edits|duration_ms|num_artists|description|
+-------------+-------------+------+-----------+----------+----------+-------------+--------------------+---------+-----------+-----------+-----------+
|          joe|        false|609000| 1465084800|        60|        59|            1|[{It's Strange, s...|        2|   13108318|         56|       NULL|
|       Summer|        false|609001| 1499904000|        42|        38|            3|[{Beauty And The ...|        5|   10348034|         31|       NULL|
| my childhood|        false|609002| 1508716800|       131|        48|            2|[{Determinate, sp...|        8|   24173095|         42|       NULL|
|           az|        false|609003| 1508803200|       105|       103|            1|[{Co

## Lets run some basic DQ checks:

In [9]:
# Are primary keys unique? Expecting 1000000 = 1000000, will return True if so

dfp.count() == dfp.select("pid").distinct().count() 

True

In [41]:
from pyspark.sql import functions as F

In [11]:
# Lets check the amount of nulls in each column - session died on first attempt to run (reconfigured session builder for this) 

dfp.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in dfp.columns
    ]
).show(truncate=False)

+----+-------------+---+-----------+----------+----------+-------------+------+---------+-----------+-----------+-----------+
|name|collaborative|pid|modified_at|num_tracks|num_albums|num_followers|tracks|num_edits|duration_ms|num_artists|description|
+----+-------------+---+-----------+----------+----------+-------------+------+---------+-----------+-----------+-----------+
|0   |0            |0  |0          |0         |0         |0            |0     |0        |0          |0          |981240     |
+----+-------------+---+-----------+----------+----------+-------------+------+---------+-----------+-----------+-----------+



In [12]:
#sanity check
dfp.select(
    [F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in dfp.columns]
).show()

+-------+-------------+-------+-----------+----------+----------+-------------+-------+---------+-----------+-----------+-----------+
|   name|collaborative|    pid|modified_at|num_tracks|num_albums|num_followers| tracks|num_edits|duration_ms|num_artists|description|
+-------+-------------+-------+-----------+----------+----------+-------------+-------+---------+-----------+-----------+-----------+
|1000000|      1000000|1000000|    1000000|   1000000|   1000000|      1000000|1000000|  1000000|    1000000|    1000000|      18760|
+-------+-------------+-------+-----------+----------+----------+-------------+-------+---------+-----------+-----------+-----------+



In [13]:
#lets count empty strings ie ""

string_cols = [c for c, t in dfp.dtypes if t == "string"]

dfp.select(
    [F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in string_cols]
).show()

+----+-------------+-----------+
|name|collaborative|description|
+----+-------------+-----------+
|   0|            0|          2|
+----+-------------+-----------+



In [14]:
dfp.filter(F.col("description") == "").show() #get PID to investigate

+-----------+-------------+------+-----------+----------+----------+-------------+--------------------+---------+-----------+-----------+-----------+
|       name|collaborative|   pid|modified_at|num_tracks|num_albums|num_followers|              tracks|num_edits|duration_ms|num_artists|description|
+-----------+-------------+------+-----------+----------+----------+-------------+--------------------+---------+-----------+-----------+-----------+
|  Indie Mix|        false|620536| 1508716800|       178|        98|            1|[{Penguin Prison,...|       51|   42541305|         85|           |
|50's & 60's|        false|101318| 1499990400|        41|        40|           10|[{The Beatles, sp...|       12|    6412867|         32|           |
+-----------+-------------+------+-----------+----------+----------+-------------+--------------------+---------+-----------+-----------+-----------+



In [15]:
#notice values are returning NOT NULL while the description is blank
dfp.filter(F.col("pid").isin(620536, 101318)).select(
    "pid",
    "description",
    F.when(F.col("description").isNotNull(), "NOT NULL")
     .otherwise("NULL")
).show(truncate=False)

+------+-----------+---------------------------------------------------------------+
|pid   |description|CASE WHEN (description IS NOT NULL) THEN NOT NULL ELSE NULL END|
+------+-----------+---------------------------------------------------------------+
|620536|           |NOT NULL                                                       |
|101318|           |NOT NULL                                                       |
+------+-----------+---------------------------------------------------------------+



In [ ]:
#Checking for any other empty strings with more than one space
dfp.select([
    F.count(F.when(F.trim(F.col(c)) == "", c)).alias(c)
    for c in string_cols
]).show()

+----+-------------+-----------+
|name|collaborative|description|
+----+-------------+-----------+
|   0|            0|          2|
+----+-------------+-----------+



In [17]:
# Empty playlists? 
dfp.filter(F.size("tracks") == 0).count()

0

In [18]:
# Playlists with one song?
dfp.filter(F.size("tracks") == 1).count()

0

In [19]:
#no mismatches between track number and actual size of track array
dfp.filter(F.col("num_tracks") != F.size("tracks")).select("pid", "num_tracks", "tracks").show()

+---+----------+------+
|pid|num_tracks|tracks|
+---+----------+------+
+---+----------+------+



In [21]:
#general view
dfp.select('modified_at', 'num_tracks', 'num_albums', 'num_followers', 'num_edits', 'duration_ms', 'num_artists').describe().show(truncate=False)

+-------+-------------------+----------------+-----------------+------------------+-----------------+--------------------+-----------------+
|summary|modified_at        |num_tracks      |num_albums       |num_followers     |num_edits        |duration_ms         |num_artists      |
+-------+-------------------+----------------+-----------------+------------------+-----------------+--------------------+-----------------+
|count  |1000000            |1000000         |1000000          |1000000           |1000000          |1000000             |1000000          |
|mean   |1.4762793727296E9  |66.346428       |49.597278        |2.597746          |17.655902        |1.5579676842698E7   |38.088211        |
|stddev |3.666991520233971E7|53.6693579991489|39.96106364797268|128.85114458295232|20.64325353984474|1.2856434575363677E7|30.28290146061965|
|min    |1271376000         |5               |2                |1                 |1                |97538               |3                |
|max    |1509

In [22]:
sample = dfp.sample(0.01).select("pid", F.explode("tracks").alias("t"))
sample.show()

+------+--------------------+
|   pid|                   t|
+------+--------------------+
|609059|{Mail On Sunday, ...|
|609059|{7/27, spotify:al...|
|609059|{MY HOUSE, spotif...|
|609059|{TALKING IS HARD,...|
|609059|{Ripcord, spotify...|
|609059|{This Is What You...|
|609059|{Listen Again, sp...|
|609059|{The Marshall Mat...|
|609059|{In A Tidal Wave ...|
|609059|{Halcyon Days, sp...|
|609059|{Somebody, spotif...|
|609059|{Ghost Stories, s...|
|609059|{FutureSex/LoveSo...|
|609059|{G I R L, spotify...|
|609059|{I'm the Man (The...|
|609059|{Lift Your Spirit...|
|609059|{Good Things, spo...|
|609059|{Watch Me (Whip /...|
|609059|{Try Everything, ...|
|609059|{The Art of Hustl...|
+------+--------------------+
only showing top 20 rows


In [23]:
#exploding so that every element of the tracks array becomes a row
exploded = dfp.select("name", "pid", F.explode('tracks').alias('tracks'))
exploded.show()

+----+------+--------------------+
|name|   pid|              tracks|
+----+------+--------------------+
| joe|609000|{It's Strange, sp...|
| joe|609000|{Me 4 U, spotify:...|
| joe|609000|{Take Me Home: Ye...|
| joe|609000|{Nothing Was The ...|
| joe|609000|{Yeezus, spotify:...|
| joe|609000|{Top Five, spotif...|
| joe|609000|{Licensed To Ill,...|
| joe|609000|{Dookie, spotify:...|
| joe|609000|{Hood Hop, spotif...|
| joe|609000|{SremmLife, spoti...|
| joe|609000|{Sweet Caroline, ...|
| joe|609000|{Don't Be S.A.F.E...|
| joe|609000|{Finally Rich, sp...|
| joe|609000|{Excuse My French...|
| joe|609000|{Rodeo, spotify:a...|
| joe|609000|{Fetty Wap, spoti...|
| joe|609000|{Jordan Belfort, ...|
| joe|609000|{Late Registratio...|
| joe|609000|{The Chainsmokers...|
| joe|609000|{Neon Future I, s...|
+----+------+--------------------+
only showing top 20 rows


In [24]:
#our tracks table! Exploding on tracks.* unnests the struct 
flat = exploded.select('name','pid', "tracks.*")
flat.show(truncate=False)

+----+------+---------------------------------------+------------------------------------+----------------+-------------------------------------+-----------+---+------------------------------------------+------------------------------------+
|name|pid   |album_name                             |album_uri                           |artist_name     |artist_uri                           |duration_ms|pos|track_name                                |track_uri                           |
+----+------+---------------------------------------+------------------------------------+----------------+-------------------------------------+-----------+---+------------------------------------------+------------------------------------+
|joe |609000|It's Strange                           |spotify:album:14qZ1kKG6UoQnBupQTtYRq|Louis The Child |spotify:artist:7wg1qvie3KqDNQbAkTdbX0|245581     |0  |It's Strange                              |spotify:track:2Th9BGKvfZG8bKQSACitwG|
|joe |609000|Me 4 U             

In [25]:
flat.printSchema()

root
 |-- name: string (nullable = true)
 |-- pid: long (nullable = true)
 |-- album_name: string (nullable = true)
 |-- album_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- pos: long (nullable = true)
 |-- track_name: string (nullable = true)
 |-- track_uri: string (nullable = true)



In [26]:
#counting the amount of track rows - Over 66 million tracks
flat.count()

66346428

In [27]:
num_cols = ['duration_ms','pos']

In [28]:
#may be a good idea to sample first before running on entire dataset
sample = flat.select(num_cols).sample(0.01).describe()
sample.show()

+-------+------------------+------------------+
|summary|       duration_ms|               pos|
+-------+------------------+------------------+
|  count|            662965|            662965|
|   mean|234691.66605175234|54.424677019148824|
| stddev|  73626.5005222391| 48.27754400002565|
|    min|                 0|                 0|
|    max|          14557544|               249|
+-------+------------------+------------------+



In [ ]:
#notice there may only really 295860 unique artists across the 1000000 playlists
flat.select("artist_uri").distinct().count()

295860

In [30]:
#multiple artists with the same name? Or artists with multiple uris? Or both?
flat.select("artist_name").distinct().count()

287742

In [31]:
flat.groupBy("artist_name") \
    .agg (F.countDistinct("artist_uri").alias ("distinct_uris")) \
    .filter (F.col ("distinct_uris") > 1) \
    .orderBy (F.col("distinct_uris").desc()) \
    .show()

+-----------+-------------+
|artist_name|distinct_uris|
+-----------+-------------+
|      Ghost|           12|
|        Kim|           11|
|        Ten|           10|
|       Luke|           10|
|     Oliver|           10|
|      Monty|            9|
|       Luna|            9|
|     Gemini|            9|
|     Joseph|            9|
|      Sasha|            9|
|      Orion|            9|
|     Apollo|            8|
|      Seven|            8|
|    Phoenix|            8|
|     Aurora|            8|
|       Alex|            8|
|      Angel|            8|
|        Joy|            8|
|    Eclipse|            7|
|    Raphael|            7|
+-----------+-------------+
only showing top 20 rows


In [32]:
#While this verifies that the distinct ghost uris are different artists, we stil don't have a way of checking if a single artist can have multiple uris
flat.filter(F.col("artist_name") == "Ghost") \
    .select("artist_name", "artist_uri") \
    .distinct() \
    .show(truncate=False)

+-----------+-------------------------------------+
|artist_name|artist_uri                           |
+-----------+-------------------------------------+
|Ghost      |spotify:artist:7sfuomvDmUbPRxzAQTRH1u|
|Ghost      |spotify:artist:64Os7ICoISNDzfoYaEVyVF|
|Ghost      |spotify:artist:3Z7ZYuUEoCMAi3k7BaQfKw|
|Ghost      |spotify:artist:7moQLt3zIV4F83NCxYVvuw|
|Ghost      |spotify:artist:6tNHdlHpPDgZQp6fa3kbM8|
|Ghost      |spotify:artist:2uSZ0CvywyP2LzmwvdKzcZ|
|Ghost      |spotify:artist:5eEJ4wJHVrUUs1th2F1TQw|
|Ghost      |spotify:artist:53AcoyTAgCiIGOv1pGuSi5|
|Ghost      |spotify:artist:4yiECzxhHtKuPaQ1erLdQy|
|Ghost      |spotify:artist:6zYtDbgziYljg4xYDaPphf|
|Ghost      |spotify:artist:7DCZiSOUt9UNnexpZ3mY2E|
|Ghost      |spotify:artist:5KnL2sD4zjIU5XKI9m5Vne|
+-----------+-------------------------------------+



In [ ]:
#notice multiple album uri for same album "Nothing Was the Same", "More Life"
flat.filter(F.col("artist_name") == "Drake") \
    .select("album_name", "artist_uri", "album_uri") \
    .distinct() \
    .orderBy("album_name") \
    .show(1000,truncate=False)

+-------------------------------------------------------------+-------------------------------------+------------------------------------+
|album_name                                                   |artist_uri                           |album_uri                           |
+-------------------------------------------------------------+-------------------------------------+------------------------------------+
|0 To 100 / The Catch Up                                      |spotify:artist:3TVXtAsR1Inumwj472S9r4|spotify:album:5OkxLN5XaE9rLbgK2FuKBE|
|5 Am in Toronto                                              |spotify:artist:3TVXtAsR1Inumwj472S9r4|spotify:album:1TLGz9wTFGvlgd85Ee9FQx|
|6 God                                                        |spotify:artist:3TVXtAsR1Inumwj472S9r4|spotify:album:4k9CtU6gKpEszlrko8hB9c|
|9AM In Dallas                                                |spotify:artist:3TVXtAsR1Inumwj472S9r4|spotify:album:5Fh3p8OsKckhgZuXQf85Im|
|Almighty                  

In [34]:
flat.filter(F.col("album_uri") == "spotify:album:2ZUFSbIkmFkGag000RWOpA" ) \
    .select("track_name", "track_uri") \
    .distinct() \
    .show(1000,truncate=False)

+-------------------------+------------------------------------+
|track_name               |track_uri                           |
+-------------------------+------------------------------------+
|Hold On, We're Going Home|spotify:track:6jdOi5U5LBzQrc4c1VT983|
|The Motion               |spotify:track:3t8pnImpBpOwxdtYBpKvA9|
+-------------------------+------------------------------------+



In [35]:
flat.filter(F.col("album_uri") == "spotify:album:2gXTTQ713nCELgPOS0qWyt" ) \
    .select("track_name", "track_uri") \
    .distinct() \
    .show(1000,truncate=False)

+---------------------------------+------------------------------------+
|track_name                       |track_uri                           |
+---------------------------------+------------------------------------+
|Worst Behavior                   |spotify:track:6oF3Es1YzzmLKjGBfThUvD|
|Come Thru                        |spotify:track:5CDy1I6rSt6vXdqIb87A6f|
|The Language                     |spotify:track:3Qu5bTS5AvgS0TpeGhQyfc|
|Pound Cake / Paris Morton Music 2|spotify:track:1HDaPtZuixue2q6VGNRdVO|
|Tuscan Leather                   |spotify:track:4IJ7ZoJ8z8cAIbqYShF3ZZ|
|Own It                           |spotify:track:3ptQ2qKjiGOIW1USCFXVtT|
|Wu-Tang Forever                  |spotify:track:6KziiQUOoCmC7Kc7Rv4jar|
|Connect                          |spotify:track:2cx10hB95ygrUp2RsZW7Oh|
|Too Much                         |spotify:track:6kIh5c8x8vzOe6OKW1X59U|
|From Time                        |spotify:track:10VBBaul4zVD0reteuIHM2|
|305 To My City                   |spotify:track:1y

## Verified on spotify to be explicit vs non-explicit albums (different album_uri explanation)

In [36]:
#looking for empty strings in tracks
tstring_cols = [c for c, t in flat.dtypes if t == "string"]
flat.select([
    F.count(F.when(F.trim(F.col(c)) == "", c)).alias(c)
    for c in tstring_cols
]).show()


+----+----------+---------+-----------+----------+----------+---------+
|name|album_name|album_uri|artist_name|artist_uri|track_name|track_uri|
+----+----------+---------+-----------+----------+----------+---------+
|   0|         0|        0|          0|         0|         0|        0|
+----+----------+---------+-----------+----------+----------+---------+



In [37]:
#null check
flat.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in flat.columns
    ]
).show(truncate=False)

+----+---+----------+---------+-----------+----------+-----------+---+----------+---------+
|name|pid|album_name|album_uri|artist_name|artist_uri|duration_ms|pos|track_name|track_uri|
+----+---+----------+---------+-----------+----------+-----------+---+----------+---------+
|0   |0  |0         |0        |0          |0         |0          |0  |0         |0        |
+----+---+----------+---------+-----------+----------+-----------+---+----------+---------+



In [52]:
# Lets start writing the tables
# Writing a function to convert empty stings to nulls using earlier logic

def emptystringconv (df, columns = None):
    """
    This function will read string columns in a dataframe and convert any whitespace string values to proper null type.
    If no columns are passed, function will loop through all string columns
    """
    if columns is None:
        for x, y in df.dtypes:
            if y == "string":
                df = df.withColumn(x, F.when(F.trim(F.col(x)) == "", F.lit(None)).otherwise(F.col(x)))

    else:
        for x in columns:
            df = df.withColumn(x, F.when(F.col(x) == "", F.lit(None)).otherwise(F.col(x)))

    return df

test = spark.read.parquet("../bronze/playlist")

test = emptystringconv(test)

test.filter(F.col("pid").isin(620536, 101318)).select(
    "pid",
    "description",
    F.when(F.col("description").isNotNull(), "NOT NULL")
     .otherwise("NULL")
).show(truncate=False)

+------+-----------+---------------------------------------------------------------+
|pid   |description|CASE WHEN (description IS NOT NULL) THEN NOT NULL ELSE NULL END|
+------+-----------+---------------------------------------------------------------+
|620536|NULL       |NULL                                                           |
|101318|NULL       |NULL                                                           |
+------+-----------+---------------------------------------------------------------+



In [53]:

playlist_t = emptystringconv(dfp)
playlist_t = playlist_t.select("pid", \
                        "name", \
                        "collaborative", \
                        "modified_at", \
                        "num_tracks", \
                        "num_albums", \
                        "num_followers", \
                        "num_edits",\
                        "num_artists", \
                        "duration_ms", \
                        "description") \
                        .withColumn("collaborative", F.col("collaborative").cast("boolean"))

playlist_t.printSchema()

root
 |-- pid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- collaborative: boolean (nullable = true)
 |-- modified_at: long (nullable = true)
 |-- num_tracks: long (nullable = true)
 |-- num_albums: long (nullable = true)
 |-- num_followers: long (nullable = true)
 |-- num_edits: long (nullable = true)
 |-- num_artists: long (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- description: string (nullable = true)



In [56]:
track_t = flat.select("track_uri",\
                "track_name", \
                "artist_uri", \
                "duration_ms") \
                .dropDuplicates(["track_uri"])

track_t.printSchema()

root
 |-- track_uri: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- duration_ms: long (nullable = true)



In [57]:
artist_t = flat.select ("artist_uri", \
                "artist_name") \
                .dropDuplicates(["artist_uri"])

artist_t.printSchema()

root
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)



In [58]:
album_t = flat.select("album_uri", \
                "album_name") \
                .dropDuplicates(['album_uri'])

album_t.printSchema()

root
 |-- album_uri: string (nullable = true)
 |-- album_name: string (nullable = true)



In [60]:
pos_bridge_t = flat.select(
    "pid", \
    "pos", \
    "track_uri"
).distinct()

pos_bridge_t.count()

66346428

In [ ]:
#took 2 minutes 20 seconds to write
playlist_t.write.parquet("../silver/playlist")
track_t.write.parquet("../silver/track")
artist_t.write.parquet("../silver/artist")
album_t.write.parquet("../silver/album")
pos_bridge_t.write.parquet("../silver/pid_pos")

In [63]:
album = spark.read.parquet("../silver/album")
track = spark.read.parquet("../silver/track")
playlist = spark.read.parquet("../silver/playlist")
artist = spark.read.parquet("../silver/artist")
pid_pos = spark.read.parquet("../silver/pid_pos")

In [ ]:
#checks out!
print(f"album count: {album.count()}")
print(f"track count: {track.count()}")
print(f"playlist count: {playlist.count()}")
print(f"artist count: {artist.count()}")
print(f"pid_pos count: {pid_pos.count()}")

album count: 734684
track count: 2262292
playlist count: 1000000
artist count: 295860
pid_pos count: 66346428
